# Chapter 8 · Quantum Support Vector Machine (QSVM)

## Objectives

1. Understand the quantum feature map as data encoding in Hilbert space.
2. Implement a quantum kernel with Qiskit Machine Learning.
3. Train and evaluate an SVM classifier with a quantum kernel on a synthetic dataset.

---

## 8.1 Quantum kernel

The quantum kernel between two points $\mathbf{x}, \mathbf{x}' \in \mathbb{R}^d$ is defined as the overlap of the states of their feature maps:

$$K(\mathbf{x}, \mathbf{x}') = |\langle \phi(\mathbf{x}) | \phi(\mathbf{x}') \rangle|^2 = |\langle 0| U^\dagger(\mathbf{x}) U(\mathbf{x}') |0\rangle|^2$$

where $U(\mathbf{x})$ is a parametric quantum circuit that depends on the data.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_aer import AerSimulator
from qiskit.primitives import Sampler

print('Quantum ML modules loaded.')

In [ ]:
# ── Dataset: two moons ───────────────────────────────────────────
np.random.seed(42)
X, y = make_moons(n_samples=100, noise=0.15)

# Normalize to [0, π]
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)

print(f'Dataset: {len(X_train)} train, {len(X_test)} test')
print(f'Features: {X.shape[1]} (2D → 2 qubits)')

# Dataset visualization
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X[y==0, 0], X[y==0, 1], c='#58a6ff', label='Class 0', alpha=0.7)
ax.scatter(X[y==1, 0], X[y==1, 1], c='#f78166', label='Class 1', alpha=0.7)
ax.set_title('Dataset: two moons (binary classification)')
ax.legend()
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e')
plt.tight_layout()
plt.show()

In [ ]:
# ── ZZFeatureMap feature map ─────────────────────────────────────
n_features = X.shape[1]   # 2
feature_map = ZZFeatureMap(
    feature_dimension=n_features,
    reps=2,
    entanglement='linear',
)

print('Quantum feature map (ZZFeatureMap):')
print(feature_map.decompose().draw('text'))

In [ ]:
# ── Quantum kernel and SVM ───────────────────────────────────────
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()
qkernel = FidelityQuantumKernel(feature_map=feature_map)

# Calculate kernel matrices
print('Calculating quantum kernel matrices...')
K_train = qkernel.evaluate(x_vec=X_train)
K_test  = qkernel.evaluate(x_vec=X_test, y_vec=X_train)

# SVM with precomputed kernel
svm_quantum = SVC(kernel='precomputed', C=1.0)
svm_quantum.fit(K_train, y_train)

# Evaluation
y_pred_q = svm_quantum.predict(K_test)
acc_q = accuracy_score(y_test, y_pred_q)

# Comparison with classical RBF SVM
svm_classic = SVC(kernel='rbf', C=1.0)
svm_classic.fit(X_train, y_train)
y_pred_c = svm_classic.predict(X_test)
acc_c = accuracy_score(y_test, y_pred_c)

print(f'\n=== Results ===')
print(f'Accuracy QSVM  (quantum ZZ kernel): {acc_q:.4f}')
print(f'Accuracy SVM   (classical RBF kernel): {acc_c:.4f}')

## 8.2 Proposed exercises

1. Try the `PauliFeatureMap` feature map with different Pauli operators. How does it affect accuracy?

2. Increase the number of reps of `ZZFeatureMap` to 3 and 4. Is there overfitting?

3. Apply QSVM to the Iris dataset (4 features, 3 classes) using a 4-qubit map.